Bioinformatics Analysis of Post-natal Day 4 mice lung samples
S7+S8
S9+S10 
Created by : Sayane Shome
Date of updated version : April 13,2022
Content :
1.Clustering of data
2.Carrying out DE analysis
3.Subsetting macrophage clusters
4.Computing RNA velocity of macrophage clusters

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv

ModuleNotFoundError: No module named 'scvelo'

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [ ]:
S7 = sc.read_10x_mtx('S7/filtered_feature_bc_matrix/',cache=True)  
S8 = sc.read_10x_mtx('S8/filtered_feature_bc_matrix/',cache=True)
S9 = sc.read_10x_mtx('S9/filtered_feature_bc_matrix/',cache=True)
S10 = sc.read_10x_mtx('S10/filtered_feature_bc_matrix/',cache=True)

In [ ]:
S7.obs['sample'] = 'Cre-'
S8.obs['sample'] = 'Cre-'
S9.obs['sample'] = 'Cre+'
S10.obs['sample'] = 'Cre+'

In [ ]:
S7.var_names_make_unique()
S8.var_names_make_unique()
S9.var_names_make_unique()
S10.var_names_make_unique()

In [ ]:
S7_loom = scv.read('S7/velocyto/possorted_genome_bam_C41I3.loom', cache=True)
S8_loom = scv.read('S8/velocyto/possorted_genome_bam_311AB.loom', cache=True)
S9_loom = scv.read('S9/velocyto/possorted_genome_bam_O24Y3.loom', cache=True)
S10_loom = scv.read('S10/velocyto/possorted_genome_bam_4HZE9.loom', cache=True)

In [ ]:
S7_1 = scv.utils.merge(S7,S7_loom)
S7 = S7_1
S8_1 = scv.utils.merge(S8,S8_loom)
S8 = S8_1
S9_1 = scv.utils.merge(S9,S9_loom)
S9 = S9_1
S10_1 = scv.utils.merge(S10,S10_loom)
S10 = S10_1


In [ ]:
sc.pl.highest_expr_genes(S7, n_top=20, )
sc.pl.highest_expr_genes(S8, n_top=20, )
sc.pl.highest_expr_genes(S9, n_top=20, )
sc.pl.highest_expr_genes(S10, n_top=20, )

In [ ]:
sc.pp.filter_cells(S7, min_genes=200)
sc.pp.filter_genes(S7, min_cells=3)
sc.pp.filter_cells(S8, min_genes=200)
sc.pp.filter_genes(S8, min_cells=3)
sc.pp.filter_cells(S9, min_genes=200)
sc.pp.filter_genes(S9, min_cells=3)
sc.pp.filter_cells(S10, min_genes=200)
sc.pp.filter_genes(S10, min_cells=3)

In [ ]:
adata = S7.concatenate(S8,S9,S10)

In [ ]:
print(adata.obs['sample'].value_counts())

In [ ]:
# mitochondrial genes
adata.var['mt'] = adata.var_names.str.startswith('mt-') 
# ribosomal genes
adata.var['ribo'] = adata.var_names.str.startswith(("Rps","Rpl"))
# hemoglobin genes.
adata.var['hb'] = adata.var_names.str.contains(("^Hb[^(p)]"))

adata.var

In [ ]:
sc.pp.calculate_qc_metrics(adata,qc_vars=['mt','ribo','hb'],percent_top=None,log1p=False,inplace=True)

In [ ]:
mito_genes = adata.var_names.str.startswith('mt-')
# for each cell compute fraction of counts in mito genes vs. all genes
# the `.A1` is only necessary as X is sparse (to transform to a dense array after summing)
adata.obs['percent_mt2'] = np.sum(
    adata[:, mito_genes].X, axis=1).A1 / np.sum(adata.X, axis=1).A1
# add the total counts per cell as observations-annotation to adata
adata.obs['n_counts'] = adata.X.sum(axis=1).A1

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20, )

In [ ]:
# filter for percent mito
adata = adata[adata.obs['pct_counts_mt'] < 16, :]

# filter for percent hb > 5
adata = adata[adata.obs['pct_counts_hb'] < 5, :]

# filter for percent ribo > 15
adata = adata[adata.obs['pct_counts_ribo'] > 15, :]

print("Remaining cells %d"%adata.n_obs)

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20, )

In [ ]:
malat1 = adata.var_names.str.startswith('Malat1')
# we need to redefine the mito_genes since they were first 
# calculated on the full object before removing low expressed genes.
mito_genes = adata.var_names.str.startswith('mt-')
hb_genes = adata.var_names.str.contains(("^Hb[^(p)]"))
rb_genes = adata.var_names.str.startswith(("Rps","Rpl"))


remove = np.add(mito_genes,malat1)
remove1 = np.add(remove,hb_genes)
remove2 = np.add(remove1,rb_genes)
keep = np.invert(remove2)

adata = adata[:,keep]

print(adata.n_obs, adata.n_vars)

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20, )

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

print(adata.n_obs, adata.n_vars)

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt','pct_counts_ribo', 'pct_counts_hb'],
             jitter=0.4, groupby = 'sample', rotation= 45)

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20, )

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)

In [ ]:
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

In [ ]:
sc.pl.highly_variable_genes(adata)

In [ ]:
adata.raw = adata

In [ ]:
adata = adata[:, adata.var.highly_variable]

In [ ]:
sc.pp.scale(adata, max_value=10)

In [ ]:
adata

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt','pct_counts_ribo', 'pct_counts_hb'],
             jitter=0.4, groupby = 'sample', rotation= 45)

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
print(adata.obs['sample'].value_counts())

In [ ]:
sc.pl.pca_variance_ratio(adata, log=True)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata)

In [ ]:
sc.tl.leiden(adata, key_added = "leiden_10") # default resolution in 1.0
sc.tl.leiden(adata, resolution = 0.3, key_added = "leiden_03")
sc.tl.leiden(adata, resolution = 0.2, key_added = "leiden_02")
sc.tl.leiden(adata, resolution = 0.1, key_added = "leiden_01")
sc.tl.leiden(adata, resolution = 0.01, key_added = "leiden_001")

#have labelled the leiden clustering without the decimal points for easier code-writing

In [ ]:
sc.pl.umap(adata, color=['leiden_03','leiden_02', 'leiden_01','leiden_10'])

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import gseapy
import matplotlib.pyplot as plt

sc.settings.verbosity = 2             # verbosity: errors (0), warnings (1), info (2), hints (3)
#sc.logging.print_versions()

In [ ]:
sc.settings.set_figure_params(dpi=80)

In [ ]:
adata = adata[:, adata.var.highly_variable]

In [ ]:
print(adata.X.shape)
print(adata.raw.X.shape)

#shows the filtered and raw version of the data

In [ ]:
adata

In [ ]:
#adata.write_h5ad('adata_afterleidenclustering_feb28_2022.h5ad')

In [ ]:
sc.tl.dendrogram(adata,'sample')

In [ ]:
sc.tl.rank_genes_groups(adata,'leiden_03', method='wilcoxon', key_added = "wilcoxon",group = "sample")
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False, key="wilcoxon")

In [ ]:
sc.pl.rank_genes_groups_heatmap(adata, n_genes=5, key="wilcoxon", groupby="sample", show_gene_labels=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata,n_genes=10,key="wilcoxon",groupby="sample")

In [ ]:
sc.pl.rank_genes_groups_stacked_violin(adata, n_genes=5, key="wilcoxon", groupby="sample")

In [ ]:
sc.tl.rank_genes_groups(adata, 'sample', groups=['Cre+'], reference='Cre-', method='wilcoxon')
sc.pl.rank_genes_groups(adata, groups=['Cre+'], n_genes=20)

In [ ]:
sc.tl.rank_genes_groups(adata,'sample', groups=['Cre+'], reference='Cre-', method='wilcoxon')
sc.pl.rank_genes_groups(adata, groups=['Cre+'], n_genes=20)

In [ ]:
sc.pl.rank_genes_groups_violin(adata,groups='Cre+', n_genes=10)

In [ ]:
sc.tl.rank_genes_groups(adata,'sample', method='wilcoxon')

# The head function returns the top n genes per cluster
top_markers = pd.DataFrame(adata.uns['rank_genes_groups']['names'])
print(top_markers)

In [ ]:
#adata.write('adata_after_diffexp.h5ad')
#adata = sc.read('adata_after_diffexp.h5ad')

In [ ]:
sc.tl.rank_genes_groups(adata,'sample', method='wilcoxon')

In [ ]:
sc.tl.rank_genes_groups(adata,'leiden_02', method='wilcoxon')

In [ ]:
top_markers = pd.DataFrame(adata.uns['rank_genes_groups']['names']).head(25)
print(top_markers)

In [ ]:
top_markers.to_csv('leiden_03.csv',index=False)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata,n_genes=10,groupby="sample")

In [ ]:
cluster2annotation = {
     '0': 'Endothelial cell',
     '1': 'Stromal Cell',
     '2': 'Alveolar Cell',
     '3': 'Macrophage/Neutrophil',
     '4': 'B-cell',
     '5': 'Stromal Cell',
     '6': 'Smooth muscle cell/Stromal Cell',
     '7': 'Stromal Cell',
     '8': 'T-cell',
    '9': 'Macrophages',
     '10': 'Alveolar Cells',
     '11': 'Smooth muscle cell',
     '12': 'Neuronal Cell',
     '13': 'Dendritic Cell',
     '14': 'Macrophages',
     '15': 'Smooth muscle cell',
     '16': 'Endothelial cell',
     '17': 'Macrophages',
      '18': 'Endothelial cell',
     '19': 'Stromal Cell',
     '20': 'Neutrophil',
     '21': 'Lung Clara cell',
     '22': 'Neutrophil',
     '23': 'Oligodendrocyte',
     '24': 'Cardiomyocyte',
     '25': 'Not decided'
}

#Created a list of manually annotated clusters and assignments done;can be used if required.

In [ ]:
#adata.obs['celltype'] = adata.obs['leiden_03'].map(cluster2annotation).astype('category')
#uncomment the previous code line to assign the clusters

In [ ]:
#sc.pl.umap(adata, color='cell type', legend_loc='on data',frameon=False, legend_fontsize=4, legend_fontoutline=2)
#uncomment the previous code line to create umap with this annotations

In [ ]:
cluster2annotation1 = {
     '0': '',
     '1': '',
     '2': '',
     '3': 'Macrophages',
     '4': '',
     '5': '',
     '6': '',
     '7': '',
     '8': '',
    '9': 'Macrophages',
     '10': '',
     '11': '',
     '12': '',
     '13': 'Macrophages',
     '14': 'Macrophages',
     '15': '',
     '16': '',
     '17': '',
      '18': '',
     '19': '',
     '20': '',
     '21': '',
     '22': 'Macrophages',
     '23': '',
     '24': '',
     '25': ''
}

In [ ]:
adata.obs['cell type_v2'] = adata.obs['leiden_03'].map(cluster2annotation1).astype('category')

In [ ]:
sc.pl.umap(adata, color='cell type_v2', legend_loc='on data',frameon=False, legend_fontsize=4, legend_fontoutline=2)

In [ ]:
sc.pl.umap(adata, color='leiden_03', legend_loc='on data',frameon=False, legend_fontsize=4, legend_fontoutline=2)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='cell type_v2', method='wilcoxon')

In [ ]:
sc.pl.rank_genes_groups(adata)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=50, groups=['Macrophages'],groupby = 'sample')

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata, n_genes=50, groups=['Macrophages'],groupby = 'sample',standard_scale='var', cmap='Blues')


In [ ]:
sc.pl.violin(adata,['Sp3'], groupby='sample')

In [ ]:
sc.pl.violin(adata,['Sp3'], groupby='leiden_001')

In [ ]:
sc.pl.umap(adata,color='leiden_001',legend_loc='on data')

In [ ]:
sc.pl.umap(adata,color='leiden_03',legend_loc='on data')

In [ ]:
sc.pl.umap(adata, color='sample')

In [ ]:
#macrophages : 3, 9, 13, 14, 22

In [ ]:
adata_sub = adata[adata.obs.leiden_03.isin(['3', '9', '13', '14', '22'])]
#subsetting the clusters represented by macrophages

In [ ]:
adata_sub = adata[adata.obs.celltype.isin(['T-cell'])]

In [ ]:
sc.pl.violin(adata_sub,['Sp3'], groupby='celltype')

In [ ]:
adata_sub.obs

In [ ]:
axs = sc.pl.rank_genes_groups_stacked_violin(adata, n_genes=50, groups=['Macrophages'],groupby = 'sample')

In [ ]:
sc.pl.violin(adata_sub,['Sp3'], groupby='sample')
#Sp3 expression in macrophages clusters

In [ ]:
sc.pl.violin(adata_sub,['Sp3'], groupby='leiden_03')
#Sp3 expression in each of the macrophage clusters

In [ ]:
sc.pl.umap(adata_sub,color = 'Sp3')

In [ ]:
sc.pl.umap(adata_sub,color = 'leiden_03')

In [ ]:
sc.tl.umap(adata_sub)

In [ ]:
sc.pl.umap(adata_sub)

In [ ]:
print(adata.obs['sample'].value_counts())

In [ ]:
print(adata_sub.obs['sample'].value_counts())

In [ ]:
sc.pl.umap(adata_sub[1:907], color='Sp3')
#Sp3 expression in Cre- samples

In [ ]:
sc.pl.umap(adata_sub[907:907+1861], color='Sp3')
#Sp3 expression in Cre+ samples

In [ ]:
chk = adata_sub[1:907].obs

In [ ]:
print(chk['sample'].value_counts())
#verifying the previous plots are made of Cre- and Cre+ samples.The next 2 steps also do the same.

In [ ]:
chk = adata_sub[907:907+1861].obs

In [ ]:
print(chk['sample'].value_counts())

In [ ]:
sc.pl.umap(adata_sub[1:1861], color='Sp3')

In [ ]:
sc.tl.dendrogram(adata_sub,groupby = 'sample')

In [ ]:
sc.pl.umap(adata_sub[1:907], color='leiden_03',frameon=False, legend_fontsize=4, legend_fontoutline=2)
#Umap based on leiden clustering of Cre- samples

In [ ]:
sc.pl.umap(adata_sub[907:907+1861], color='leiden_03',frameon=False, legend_fontsize=4, legend_fontoutline=2)
#Umap based on leiden clustering of Cre+ samples

In [ ]:
#adata_sub.obs['cell type_v2'] = 'Macrophages'

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_sub, n_genes=100, groups=['Macrophages'],groupby = 'sample')

In [ ]:
sc.pl.rank_genes_groups_stacked_violin(adata_sub, n_genes=100, groups=['Macrophages'],groupby = 'sample')

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata_sub, n_genes=100, groups=['Macrophages'],groupby = 'sample',standard_scale='var', cmap='Blues')

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata_sub, n_genes=100, groups=['Macrophages'],groupby = 'leiden_03',standard_scale='var', cmap='Blues')

In [ ]:
sc.tl.rank_genes_groups(adata_sub, groupby='leiden_03', method='wilcoxon')

In [ ]:
sc.pl.rank_genes_groups(adata_sub, groupby='leiden_03', method='wilcoxon')

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata_sub, n_genes=40, groups=['22'],groupby = 'sample',standard_scale='var', cmap='Blues')

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata_sub, n_genes=40, groups=['3'],groupby = 'sample',standard_scale='var', cmap='Blues')

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata_sub, n_genes=40, groups=['9'],groupby = 'sample',standard_scale='var', cmap='Blues')

In [ ]:
sc.pl.umap(adata_sub,color = 'leiden_03')

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata_sub, n_genes=40, groups=['14'],groupby = 'sample',standard_scale='var', cmap='Blues')

In [ ]:
scv.set_figure_params()

In [ ]:
#Computing RNA velocity for macrophage clusters
scv.pl.proportions(adata_sub)

In [ ]:
scv.pp.filter_genes(adata_sub, min_shared_counts=20)
scv.pp.normalize_per_cell(adata_sub)
scv.pp.filter_genes_dispersion(adata_sub, n_top_genes=2000)
scv.pp.log1p(adata_sub)

In [ ]:
scv.pp.filter_and_normalize(adata_sub, min_shared_counts=20, n_top_genes=2000)
scv.pp.moments(adata_sub, n_pcs=30, n_neighbors=30)

In [ ]:
scv.tl.velocity(adata_sub)

In [ ]:
scv.tl.velocity_graph(adata_sub)

In [ ]:
scv.pl.velocity_embedding_stream(adata_sub, basis='umap',color = 'leiden_03')

In [ ]:
scv.pl.velocity_embedding(adata_sub, arrow_length=3, arrow_size=2, dpi=120,color = 'leiden_03')

In [ ]:
scv.tl.score_genes_cell_cycle(adata_sub)
scv.pl.scatter(adata_sub, color_gradients=['S_score', 'G2M_score'], smooth=True, perc=[5, 95])

In [ ]:
scv.tl.velocity_confidence(adata_sub)
keys = 'velocity_length', 'velocity_confidence'
scv.pl.scatter(adata_sub, c=keys, cmap='coolwarm', perc=[5, 95])

In [ ]:
scv.pl.velocity_graph(adata_sub, threshold=.1,color = 'leiden_03')

In [ ]:
x, y = scv.utils.get_cell_transitions(adata_sub, basis='umap', starting_cell=70)
ax = scv.pl.velocity_graph(adata_sub, c='lightgrey', edge_width=.05, show=False)
ax = scv.pl.scatter(adata_sub, x=x, y=y, s=120, c='ascending', cmap='gnuplot', ax=ax)

In [ ]:
scv.tl.velocity_pseudotime(adata_sub)
scv.pl.scatter(adata_sub, color='velocity_pseudotime', cmap='gnuplot')

In [ ]:
scv.pl.proportions(adata)

In [ ]:
scv.pp.filter_genes(adata, min_shared_counts=20)
scv.pp.normalize_per_cell(adata)
scv.pp.filter_genes_dispersion(adata, n_top_genes=2000)
scv.pp.log1p(adata)

In [ ]:
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

In [ ]:
scv.tl.velocity(adata)

In [ ]:
scv.tl.velocity_graph(adata)

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='umap',color = 'cell type_v2')

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('Edited_PanglaoDB.csv')
#Reading edited PanglaoDB(https://academic.oup.com/database/article/doi/10.1093/database/baz046/5427041)

In [ ]:
df['official.gene.symbol'] = df['official.gene.symbol'].str.lower()
df['official.gene.symbol'] = df['official.gene.symbol'].str.capitalize()


In [ ]:
monocytes = df.loc[df['cell.type'] == 'Monocytes']

In [ ]:
alveolar_macrophages = df.loc[df['cell.type'] == 'Alveolar macrophages']

In [ ]:
dendritic_cells = df.loc[df['cell.type'] == 'Dendritic cells']

In [ ]:

monocytes_markers = list(monocytes['official.gene.symbol'])

In [ ]:
dendritic_cell_markers = list(dendritic_cells['official.gene.symbol'])

In [ ]:
alveolar_macrophages_markers = list(alveolar_macrophages['official.gene.symbol'])

In [ ]:
monocytes_markers = list(monocytes['official.gene.symbol'])

In [ ]:
#common_list = set(list_one).intersection(list_two)

In [ ]:
s = pd.DataFrame(adata.var)

In [ ]:
s.index.name = 'gene_name'
s.reset_index(inplace=True)

In [ ]:
s['gene_name'] = s['gene_name'].str.lower()
dataset_genes = list(s['gene_name'].str.capitalize())



In [ ]:
def intersection(lst1, lst2):
    lst3 = [value for value in lst1 if value in lst2]
    return lst3
#Finding monocyte markers,alveolar macrophages,dendritic cells 
mono_common_list = intersection(monocytes_markers,dataset_genes)
dendr_common_list = intersection(dendritic_cell_markers,dataset_genes)
alveo_common_list = intersection(alveolar_macrophages_markers,dataset_genes)

#three genes in dendritic list is giving issues in the code;if we really really need them,I can include them in. 
elements = ['H2-ab1', 'H2-dma', 'H2-eb1']
dendr_common_list1 = list(set(dendr_common_list) - set(elements))


In [ ]:
sc.pl.heatmap(adata,mono_common_list,groupby='sample',swap_axes=True)

In [ ]:
sc.pl.heatmap(adata,dendr_common_list1,groupby='sample',show_gene_labels=True,swap_axes=True)

In [ ]:
sc.pl.heatmap(adata,alveo_common_list,groupby='sample',swap_axes=True)

In [ ]:
sc.pl.heatmap(adata_sub,alveo_common_list,groupby='leiden_03',swap_axes=True)

In [ ]:
sc.pl.heatmap(adata,alveo_common_list,groupby='sample',swap_axes=True)

In [ ]:
sc.pl.heatmap(adata_sub,mono_common_list,groupby='leiden_03',swap_axes=True)

In [ ]:
sc.pl.heatmap(adata_sub,dendr_common_list1,groupby='leiden_03',show_gene_labels=True,swap_axes=True)

In [ ]:
sc.pl.violin(adata,['Plet1'], groupby='sample')

In [ ]:
sc.pl.violin(adata,['Cxcl2'], groupby='sample')

In [ ]:
sc.pl.violin(adata,['Plac8'], groupby='sample')

In [ ]:
sc.pl.violin(adata,['Cr2'], groupby='sample')

In [ ]:
sc.tl.rank_genes_groups(adata_sub,'sample', groups=['Cre+'], reference='Cre-', method='wilcoxon')
sc.pl.rank_genes_groups(adata_sub, groups=['Cre+'], n_genes=50)

In [ ]:
sc.pl.violin(adata_sub,['Cd9'], groupby='sample')

In [ ]:
sc.pl.violin(adata,['S100a4'], groupby='sample')

In [ ]:
sc.pl.violin(adata,['Cd63'], groupby='sample')

In [ ]:
sc.pl.violin(adata_sub,['Calm1'], groupby='sample')

In [ ]:
sc.pl.violin(adata_sub,['Nfkb1'], groupby='sample')

In [ ]:
sc.pl.violin(adata_sub,['Ikbkb'], groupby='sample')

In [ ]:
sc.pl.rank_genes_groups(adata_sub,groupby='sample', method='wilcoxon',n_genes=50)

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata_sub, n_genes=40,groupby = 'sample',standard_scale='var', cmap='Blues')

In [ ]:
sc.pl.dotplot(adata_sub,['Calm1','Cox4i1','Cox7c','Scand1','Smdt1','Txn1','Tyrobp','H2-D1'],groupby='sample')

In [ ]:
sc.pl.dotplot(adata_sub,['Tyrobp','H2-D1'],groupby='sample')

In [ ]:
sc.pl.dotplot(adata_sub,['Tyrobp'],groupby='sample')

In [ ]:
sc.pl.dotplot(adata_sub,['H2-D1'],groupby='sample')

In [ ]:
sc.pl.dotplot(adata_sub,['Arpc4'],groupby='sample')

In [ ]:
sample1 = ['Calm1','Cox4i1','Cox7c','Scand1','Smdt1','Txn1','Arpc4','Tyrobp','H2-D1','Lars2']
sc.pl.heatmap(adata_sub,sample1,groupby='sample',show_gene_labels=True)

In [ ]:
sc.pl.dotplot(adata_sub,['C1qa'],groupby='sample')

In [ ]:
sc.pl.dotplot(adata_sub,['Ikbkb'],groupby='sample')

In [ ]:
sc.pl.dotplot(adata_sub,['Sp3'],groupby='sample')

In [ ]:
sc.pl.dotplot(adata_sub,['Sp2'],groupby='sample')

In [ ]:
result = adata_sub.uns['rank_genes_groups']
groups = result['names'].dtype.names
s = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'pvals_adj','logfoldchanges']})

In [ ]:
adata_sub

In [ ]:
s

In [ ]:
sc.pl.rank_genes_groups(adata_sub, n_genes=20)

In [ ]:
s['log10_value'] = np.log10(s['Cre+_p'])*(-1)
print(s)


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

In [ ]:
s['minus_log10_pvalue'] = -np.log10(s['Cre+_p'])
s['logFC'] = s['Cre+_l']

In [ ]:
s1 = s.loc[(s['logFC'] > -10) & (s['logFC'] < 10)]

In [ ]:
s2 = s1.loc[(s['minus_log10_pvalue'] > 0)]

In [ ]:
s1 = s2

In [ ]:
fig = go.Figure()
trace1 = go.Scatter(x=s1['logFC'],y=s1['minus_log10_pvalue'],mode='markers',hovertext=s1['Cre+_n'],text=s1['Cre+_n'])
fig.add_trace(trace1)

In [ ]:
s1.to_csv('DEG_cutoff_macrophageclusters_.csv')

In [ ]:
#Checking all DE genes 

result1 = adata.uns['rank_genes_groups']
groups = result1['names'].dtype.names
s12 = pd.DataFrame(
    {group + '_' + key[:1]: result1[key][group]
    for group in groups for key in ['names', 'pvals_adj','logfoldchanges']})

In [ ]:
s12 

In [ ]:
groups

In [ ]:
s

In [ ]:
s.to_csv('DEG_cutoff_macrophageclusters_full_for_combine_jupyternotebook.csv')